# Task 1 — QuadX-Hover-v4 with PPO

**Goal.** Train a PPO policy to hold a quadrotor stationary at the origin
with a level attitude, in PyFlyt's `QuadX-Hover-v4` environment under
**flight mode 0** (angular-rate + thrust commands).

This notebook does the full pipeline:

1. Configure environment, hyperparameters, and seeds.
2. Train PPO across 5 independent seeds (Stable-Baselines3).
3. Aggregate the per-seed evaluation curves logged during training.
4. Plot learning curves with **bootstrap confidence intervals** (Agarwal et al. 2021 style).
5. Re-evaluate each final checkpoint over 20 deterministic episodes
   using the same protocol as `scripts/evaluate.py`.
6. Report aggregate statistics: mean, IQM, bootstrap 95 % CI, crash rate.

**Reproducibility.** Seeds are propagated to Python `random`, NumPy, PyTorch,
and to the environment via `env.reset(seed=...)`. Training is idempotent: a
checkpoint that already exists on disk is skipped unless `FORCE_RETRAIN = True`.

**Compute.** ~50 minutes on a modern CPU (5 seeds × 500k env steps × 4 vec-envs).
GPU is *not* required for an MLP policy on a 12-D observation; SB3 will use
CUDA if available but the speed-up is marginal here.

In [ ]:
import sys

# Install missing dependencies for the QuadX-Hover-v4 task
!pip install PyFlyt stable-baselines3 gymnasium shimmy

## 1. Imports and paths

In [ ]:
from __future__ import annotations  # forward-compat for X | Y type hints on Python 3.9

import json
import os
import random
import sys
from pathlib import Path

import gymnasium
import numpy as np
import torch

# PyFlyt registers its envs as a side-effect of this import.
import PyFlyt.gym_envs  # noqa: F401

import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

# We use the same env_config/wrappers convention as the provided scripts/.
PROJECT_ROOT = Path.cwd().resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

# These are not strictly needed for Hover (Hover is not a Dict-obs env),
# but importing keeps the notebook drop-in compatible with the same layout
# we will reuse for Waypoints later.
try:
    from env_config import get_env_kwargs  # noqa: F401
    from wrappers import FlattenWaypointEnv  # noqa: F401
except ImportError as e:
    print(f"[warn] could not import scripts.env_config/wrappers: {e}")
    print("       This is fine for Hover, but make sure scripts/ exists for Waypoints.")

print(f"Stable-Baselines3 version: {sb3.__version__}")
print(f"PyTorch version:           {torch.__version__}")
print(f"CUDA available:            {torch.cuda.is_available()}")
print(f"Project root:              {PROJECT_ROOT}")


## 2. Configuration

All hyperparameters live in one place. `SEEDS` controls the number of independent
training runs aggregated in the statistical analysis.

In [ ]:
# ---- Experiment identity ---------------------------------------------------
ENV_ID         = "PyFlyt/QuadX-Hover-v4"
ENV_SHORT      = "QuadX-Hover-v4"      # used in filenames to match scripts/evaluate.py
ALGO_NAME      = "PPO"
FLIGHT_MODE    = 0                      # angular rate + thrust (the standard Hover MDP)

# ---- Compute budget --------------------------------------------------------
SEEDS          = [0, 1, 2, 3, 4]        # 5 seeds → bootstrap CIs are honest
TOTAL_TIMESTEPS = 500_000               # per seed
N_ENVS         = 4                      # parallel envs per training run
EVAL_FREQ      = 10_000                 # eval every 10k timesteps (counted on the *vec* env)
N_EVAL_EPISODES = 10                    # episodes per evaluation point during training
FINAL_EVAL_EPISODES = 20                # episodes for the post-training final evaluation
                                        # (matches scripts/evaluate.py default)

# ---- PPO hyperparameters (close to SB3 defaults for continuous control) ----
PPO_KWARGS = dict(
    policy="MlpPolicy",
    learning_rate=3e-4,
    n_steps=2048,           # rollout length per env → 2048 * N_ENVS = 8192 transitions / update
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])),
    verbose=0,
)

# ---- Output paths (mirror scripts/evaluate.py conventions) -----------------
RESULTS_DIR    = PROJECT_ROOT / "results"
MODELS_DIR     = RESULTS_DIR / "models"
LOGS_DIR       = RESULTS_DIR / "logs"
EVAL_DIR       = RESULTS_DIR / "eval"
FIGURES_DIR    = RESULTS_DIR / "figures"
for d in (MODELS_DIR, LOGS_DIR, EVAL_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Behaviour flags -------------------------------------------------------
FORCE_RETRAIN  = False                  # set True to overwrite existing checkpoints
USE_WANDB      = False                  # set True (and have WANDB_API_KEY set) to log to W&B
WANDB_PROJECT  = "info8003-rl-pyflyt"
WANDB_ENTITY   = None                   # your W&B username/team, or None for default

def run_name(algo: str, env_short: str, seed: int) -> str:
    return f"{algo}_{env_short}_seed{seed}"

def model_path(algo: str, env_short: str, seed: int) -> Path:
    # scripts/evaluate.py loads "...zip" via PPO.load(model_path); model.save(p) writes p.zip.
    return MODELS_DIR / f"final_{run_name(algo, env_short, seed)}"


## 3. Environment factory

We wrap each env with `Monitor` so that SB3 can log episode returns and lengths,
and we explicitly seed `env.reset` so that the initial-state distribution is
deterministic per (seed × env-rank) pair.

In [ ]:
def make_env(seed: int, flight_mode: int = FLIGHT_MODE, render_mode=None,
             monitor_path: str | None = None):
    """Return a thunk that builds one Hover env. Use with DummyVecEnv/SubprocVecEnv."""
    def _init():
        env = gymnasium.make(ENV_ID, flight_mode=flight_mode, render_mode=render_mode)
        # Monitor records episode returns/lengths into a CSV when monitor_path is set.
        env = Monitor(env, filename=monitor_path)
        # Seed the env's reset RNG; SB3 will also seed action_space at vec-env level.
        env.reset(seed=seed)
        env.action_space.seed(seed)
        return env
    return _init


def make_vec_env(base_seed: int, n_envs: int = N_ENVS):
    """Make a DummyVecEnv with `n_envs` parallel Hover envs, deterministically seeded.

    We use DummyVecEnv (not SubprocVecEnv) because PyBullet can be unstable across
    fork/spawn boundaries, and the speed-up of subprocess parallelism is modest
    here (the bottleneck is the Python physics step, not the policy network).
    """
    fns = [make_env(seed=base_seed + i) for i in range(n_envs)]
    vec = DummyVecEnv(fns)
    # VecMonitor on top of per-env Monitors gives us a vec-level episode buffer
    # which the SB3 logger uses for "rollout/ep_rew_mean".
    vec = VecMonitor(vec)
    return vec


## 4. Training one seed

Training is wrapped in a single function so we can call it in a loop, in
parallel, or skip it for seeds whose checkpoint already exists. The function
is **idempotent**: rerunning it with the same seed regenerates the same model.

During training we log to:

* TensorBoard (always),
* an SB3 `EvalCallback` that periodically rolls out the deterministic policy
  and writes `evaluations.npz` — this is what we plot below,
* Weights & Biases (only if `USE_WANDB=True` and the package is installed).

In [ ]:
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, PyTorch and SB3 in one call."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_random_seed(seed)


def train_one_seed(seed: int,
                   total_timesteps: int = TOTAL_TIMESTEPS,
                   force_retrain: bool = FORCE_RETRAIN,
                   use_wandb: bool = USE_WANDB) -> Path:
    """Train one PPO run on Hover; return the path to the saved checkpoint (without .zip)."""

    name      = run_name(ALGO_NAME, ENV_SHORT, seed)
    ckpt_path = model_path(ALGO_NAME, ENV_SHORT, seed)
    log_path  = LOGS_DIR / name
    log_path.mkdir(parents=True, exist_ok=True)

    if ckpt_path.with_suffix(".zip").exists() and not force_retrain:
        print(f"[skip] {name}: checkpoint exists, set FORCE_RETRAIN=True to overwrite.")
        return ckpt_path

    print(f"[train] {name}: {total_timesteps:,} timesteps")
    seed_everything(seed)

    # ---- Envs ----
    train_env = make_vec_env(base_seed=seed, n_envs=N_ENVS)
    # Eval env: a single deterministic env, with a *different* seed so eval is not in-distribution
    # with the most recent rollout buffer.
    eval_env  = make_vec_env(base_seed=seed + 10_000, n_envs=1)

    # ---- W&B (optional) ----
    wandb_run = None
    callbacks: list = []
    if use_wandb:
        try:
            import wandb
            from wandb.integration.sb3 import WandbCallback
            wandb_run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name,
                config={**PPO_KWARGS, "seed": seed, "total_timesteps": total_timesteps,
                        "env_id": ENV_ID, "flight_mode": FLIGHT_MODE, "n_envs": N_ENVS},
                sync_tensorboard=True, monitor_gym=False, save_code=False,
                reinit=True,
            )
            callbacks.append(WandbCallback(verbose=0))
        except Exception as e:
            print(f"[warn] W&B disabled ({e})")
            wandb_run = None

    # ---- Eval callback ----
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(log_path / "best"),
        log_path=str(log_path),                     # writes evaluations.npz
        eval_freq=max(EVAL_FREQ // N_ENVS, 1),       # SB3 counts in *vec-env* steps
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        render=False,
    )
    callbacks.append(eval_cb)

    # ---- Train ----
    model = PPO(env=train_env, seed=seed, tensorboard_log=str(log_path / "tb"), **PPO_KWARGS)
    model.learn(total_timesteps=total_timesteps, callback=callbacks, progress_bar=True)
    model.save(str(ckpt_path))
    print(f"[save] {ckpt_path}.zip")

    train_env.close(); eval_env.close()
    if wandb_run is not None:
        wandb_run.finish()
    return ckpt_path


## 5. Run training across all seeds

Sequential by default (so the progress bar stays readable). On a multi-core
machine you can parallelise this loop with `joblib` or `multiprocessing.Pool`,
but be aware that each PPO run already uses `N_ENVS=4` parallel envs.

In [ ]:
checkpoint_paths: list[Path] = []
for seed in SEEDS:
    ckpt = train_one_seed(seed)
    checkpoint_paths.append(ckpt)

print("\nAll checkpoints:")
for p in checkpoint_paths:
    print(f"  {p}.zip   exists={p.with_suffix('.zip').exists()}")


## 6. Aggregate per-seed evaluation logs

`EvalCallback` writes one `evaluations.npz` per seed. Each file contains:

* `timesteps`: shape `(n_evals,)` — env steps at which evaluation was performed,
* `results`: shape `(n_evals, N_EVAL_EPISODES)` — episode returns,
* `ep_lengths`: shape `(n_evals, N_EVAL_EPISODES)`.

We collapse across episodes (mean per evaluation point), giving one curve per seed.

In [ ]:
def load_eval_curves(seeds=SEEDS):
    """Return (timesteps, returns) where returns has shape (n_seeds, n_evals)."""
    all_returns, ref_ts = [], None
    for s in seeds:
        path = LOGS_DIR / run_name(ALGO_NAME, ENV_SHORT, s) / "evaluations.npz"
        if not path.exists():
            print(f"[warn] missing eval log for seed {s}: {path}")
            continue
        data = np.load(path)
        ts, results = data["timesteps"], data["results"]   # results shape: (n_evals, n_eps)
        per_eval_mean = results.mean(axis=1)
        all_returns.append(per_eval_mean)
        if ref_ts is None:
            ref_ts = ts
        elif len(ts) != len(ref_ts):
            # Shouldn't happen with the same EVAL_FREQ and TOTAL_TIMESTEPS, but be defensive.
            n = min(len(ts), len(ref_ts))
            ref_ts = ref_ts[:n]
            all_returns = [r[:n] for r in all_returns] + [per_eval_mean[:n]]
            all_returns.pop(-2)
    if not all_returns:
        raise RuntimeError("No eval logs found. Did training complete?")
    return ref_ts, np.stack(all_returns, axis=0)


timesteps, returns_per_seed = load_eval_curves()
print(f"timesteps:        {timesteps.shape}")
print(f"returns_per_seed: {returns_per_seed.shape}  (n_seeds, n_evals)")
print(f"Final-eval mean across seeds: {returns_per_seed[:, -1].mean():.2f}"
      f" ± {returns_per_seed[:, -1].std():.2f}")


## 7. Learning curves with bootstrap confidence intervals

Following Agarwal *et al.* (2021), we report a **bootstrap 95 % CI** of the
mean across seeds at each evaluation point, rather than mean ± std. The
bootstrap is more honest about the small-sample uncertainty of 5 seeds.

In [ ]:
import matplotlib.pyplot as plt


def bootstrap_ci(x: np.ndarray, n_boot: int = 5000, ci: float = 95.0,
                 stat=np.mean, rng: np.random.Generator | None = None):
    """Return (low, high) percentile-bootstrap CI of `stat(x)` over axis 0."""
    if rng is None:
        rng = np.random.default_rng(0)
    n = x.shape[0]
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = stat(x[idx], axis=0)
    alpha = (100.0 - ci) / 2.0
    return np.percentile(boot_stats, alpha), np.percentile(boot_stats, 100 - alpha)


def curve_with_ci(returns: np.ndarray, n_boot: int = 2000):
    """Compute mean + bootstrap CI at each evaluation point.

    returns: (n_seeds, n_evals)
    Output: (mean, ci_low, ci_high) each of shape (n_evals,).
    """
    rng = np.random.default_rng(0)
    n_seeds, n_evals = returns.shape
    mean = returns.mean(axis=0)
    low  = np.empty(n_evals)
    high = np.empty(n_evals)
    for j in range(n_evals):
        low[j], high[j] = bootstrap_ci(returns[:, j], n_boot=n_boot, rng=rng)
    return mean, low, high


mean_curve, ci_low, ci_high = curve_with_ci(returns_per_seed)

fig, ax = plt.subplots(figsize=(8, 5))
# Per-seed thin lines for transparency about variance.
for i, s in enumerate(SEEDS):
    ax.plot(timesteps, returns_per_seed[i], alpha=0.25, lw=1, label=f"seed {s}" if i == 0 else None)
ax.plot(timesteps, mean_curve, lw=2.2, color="C0", label="mean across seeds")
ax.fill_between(timesteps, ci_low, ci_high, alpha=0.20, color="C0", label="95 % bootstrap CI")
ax.set_xlabel("Environment steps")
ax.set_ylabel("Mean evaluation return")
ax.set_title(f"{ALGO_NAME} on {ENV_SHORT} (flight mode {FLIGHT_MODE}, {len(SEEDS)} seeds)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()

fig_path = FIGURES_DIR / f"learning_curve_{ALGO_NAME}_{ENV_SHORT}_mode{FLIGHT_MODE}.png"
fig.savefig(fig_path, dpi=150)
print(f"Saved figure: {fig_path}")
plt.show()


## 8. Final evaluation (matches `scripts/evaluate.py` protocol)

We re-evaluate each final checkpoint over `FINAL_EVAL_EPISODES=20` deterministic
rollouts with seeds `100..119`, exactly as `scripts/evaluate.py` would do it.
This gives a single per-seed mean return that we use for the aggregate statistics.

In [ ]:
def evaluate_checkpoint(ckpt_no_ext: Path, n_episodes: int = FINAL_EVAL_EPISODES,
                        flight_mode: int = FLIGHT_MODE) -> dict:
    """Re-implements scripts/evaluate.py.evaluate_model() in-process so we can plot."""
    model = PPO.load(str(ckpt_no_ext))
    env = gymnasium.make(ENV_ID, flight_mode=flight_mode)

    ep_returns, ep_lengths, ep_crashed = [], [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=100 + i)
        total, steps, crashed = 0.0, 0, False
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = env.step(action)
            total += r; steps += 1
            if terminated:
                crashed = r <= -50           # same heuristic as scripts/evaluate.py
                break
            if truncated:
                break
        ep_returns.append(total); ep_lengths.append(steps); ep_crashed.append(crashed)
    env.close()
    return {
        "checkpoint": str(ckpt_no_ext),
        "n_episodes": n_episodes,
        "ep_returns": [float(x) for x in ep_returns],
        "mean_return": float(np.mean(ep_returns)),
        "std_return":  float(np.std(ep_returns)),
        "median_return": float(np.median(ep_returns)),
        "mean_length": float(np.mean(ep_lengths)),
        "crash_rate":  float(np.mean(ep_crashed)),
    }


final_results = {}
for seed, ckpt in zip(SEEDS, checkpoint_paths):
    print(f"[eval] seed={seed}")
    res = evaluate_checkpoint(ckpt)
    final_results[seed] = res
    out = EVAL_DIR / f"{run_name(ALGO_NAME, ENV_SHORT, seed)}.json"
    with open(out, "w") as f:
        json.dump(res, f, indent=2)
    print(f"  mean={res['mean_return']:.2f}  std={res['std_return']:.2f}"
          f"  crash_rate={res['crash_rate']*100:.0f}%  len={res['mean_length']:.0f}")


## 9. Aggregate statistics across seeds

Three numbers, all computed on the per-seed *mean* final return:

* **Mean ± std**: the textbook number, included for completeness.
* **IQM** (interquartile mean): mean of the middle 50 % of the values.
  Recommended by Agarwal *et al.* as more robust to single-seed outliers.
* **Bootstrap 95 % CI of the mean**: honest small-sample uncertainty.

We also compute these on the *concatenated* per-episode returns (5 × 20 = 100 episodes)
for an episode-level CI.

In [ ]:
per_seed_means = np.array([final_results[s]["mean_return"] for s in SEEDS])
all_episode_returns = np.concatenate([final_results[s]["ep_returns"] for s in SEEDS])
crash_rates = np.array([final_results[s]["crash_rate"] for s in SEEDS])


def iqm(x: np.ndarray) -> float:
    q1, q3 = np.percentile(x, [25, 75])
    middle = x[(x >= q1) & (x <= q3)]
    return float(middle.mean()) if len(middle) else float(x.mean())


lo_seed, hi_seed = bootstrap_ci(per_seed_means)
lo_ep,   hi_ep   = bootstrap_ci(all_episode_returns)

print(f"=== {ALGO_NAME} on {ENV_SHORT} (flight mode {FLIGHT_MODE}) ===")
print(f"Seeds: {SEEDS}")
print(f"Per-seed mean returns: {per_seed_means.round(2).tolist()}")
print()
print("Across-seed statistics (n=5 means):")
print(f"  Mean    : {per_seed_means.mean():7.2f}  ± {per_seed_means.std():.2f} (std)")
print(f"  IQM     : {iqm(per_seed_means):7.2f}")
print(f"  95 % CI : [{lo_seed:7.2f}, {hi_seed:7.2f}]   (bootstrap, n_boot=5000)")
print()
print(f"Across-episode statistics (n={len(all_episode_returns)} episodes):")
print(f"  Mean    : {all_episode_returns.mean():7.2f}  ± {all_episode_returns.std():.2f} (std)")
print(f"  IQM     : {iqm(all_episode_returns):7.2f}")
print(f"  95 % CI : [{lo_ep:7.2f}, {hi_ep:7.2f}]")
print()
print(f"Crash rate: mean={crash_rates.mean()*100:.1f}%  per-seed={[f'{c*100:.0f}%' for c in crash_rates]}")

summary = {
    "algo": ALGO_NAME, "env": ENV_SHORT, "flight_mode": FLIGHT_MODE,
    "seeds": SEEDS, "per_seed_means": per_seed_means.tolist(),
    "across_seed": {"mean": float(per_seed_means.mean()),
                    "std": float(per_seed_means.std()),
                    "iqm": iqm(per_seed_means),
                    "ci95_low": float(lo_seed), "ci95_high": float(hi_seed)},
    "across_episode": {"mean": float(all_episode_returns.mean()),
                       "std": float(all_episode_returns.std()),
                       "iqm": iqm(all_episode_returns),
                       "ci95_low": float(lo_ep), "ci95_high": float(hi_ep)},
    "crash_rate_mean": float(crash_rates.mean()),
}
with open(EVAL_DIR / f"summary_{ALGO_NAME}_{ENV_SHORT}_mode{FLIGHT_MODE}.json", "w") as f:
    json.dump(summary, f, indent=2)


## 10. Per-episode return distribution

A box-plot per seed shows whether the variability we see across seeds is dominated
by between-seed differences in the policy or by within-seed episode-to-episode noise.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
data = [final_results[s]["ep_returns"] for s in SEEDS]
ax.boxplot(data, labels=[f"seed {s}" for s in SEEDS], showmeans=True)
ax.set_ylabel("Episode return")
ax.set_title(f"{ALGO_NAME} on {ENV_SHORT} — final eval ({FINAL_EVAL_EPISODES} episodes per seed)")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"final_eval_box_{ALGO_NAME}_{ENV_SHORT}.png", dpi=150)
plt.show()


## 11. Optional — render a deterministic rollout

Instantiates a single env with `render_mode="human"` and rolls out one episode
from the best seed. This requires a display (X / X-forwarding); skip on a headless
server. The cell is guarded so re-running the notebook end-to-end won't open a window
unless you flip the flag.

In [ ]:
DO_RENDER = False  # flip to True locally if you have a display

if DO_RENDER:
    best_seed = SEEDS[int(np.argmax(per_seed_means))]
    print(f"Rendering with seed {best_seed} (best by mean final return)")
    model = PPO.load(str(model_path(ALGO_NAME, ENV_SHORT, best_seed)))
    env = gymnasium.make(ENV_ID, flight_mode=FLIGHT_MODE, render_mode="human")
    obs, _ = env.reset(seed=42)
    total = 0.0; steps = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(action)
        total += r; steps += 1
        if terminated or truncated:
            break
    env.close()
    print(f"Rendered episode: return={total:.2f}, length={steps}")
else:
    print("Rendering skipped (DO_RENDER=False).")


## 12. Outputs

After running this notebook end-to-end you will have:

* `results/models/final_PPO_QuadX-Hover-v4_seed{0..4}.zip` — final checkpoints,
  loadable by `scripts/evaluate.py` and reusable in the dogfight tournament wrapper.
* `results/logs/PPO_QuadX-Hover-v4_seed{0..4}/evaluations.npz` — per-seed eval curves.
* `results/eval/PPO_QuadX-Hover-v4_seed{0..4}.json` — final 20-episode eval per seed.
* `results/eval/summary_PPO_QuadX-Hover-v4_mode0.json` — aggregate statistics.
* `results/figures/learning_curve_PPO_QuadX-Hover-v4_mode0.png` — mean ± bootstrap CI.
* `results/figures/final_eval_box_PPO_QuadX-Hover-v4.png` — per-seed return distributions.

The numbers in `summary_*.json` are the ones to drop into Section 4
(*Experiments & Results*) of the report.